In [1]:
import numpy as np
import sklearn
import torch
import os
from typing import OrderedDict

from torch.utils.data import TensorDataset, DataLoader

In [2]:
if not os.path.exists('tree_species_classifier_data.npz'):
  !wget -O tree_species_classifier_data.npz "https://www.dropbox.com/scl/fi/b7mw23k3ifaeui9m8nnn3/tree_species_classifier_data.npz?rlkey=bgxp37c1t04i7q35waf3slc26&dl=1"

In [3]:
data = np.load('tree_species_classifier_data.npz')
train_features = data['train_features']
train_labels = data['train_labels']
test_features = data['test_features']
test_labels = data['test_labels']

In [4]:
# examine dataset shape and train/test split
print(f'Train Features Shape: {train_features.shape} of type {train_features.dtype}')
print(f'Train Labels Shape: {train_labels.shape} of type {train_labels.dtype}')
print(f'Test Features Shape: {test_features.shape} of type {test_features.dtype}')
print(f'Test Labels Shape: {test_labels.shape} of type {test_labels.dtype}')

Train Features Shape: (15707, 426) of type int16
Train Labels Shape: (15707,) of type uint8
Test Features Shape: (1554, 426) of type int16
Test Labels Shape: (1554,) of type uint8


In [5]:
np.max(train_features, axis=1)
min(np.max(train_features, axis=1))
# max(np.min(train_features, axis=1))

np.int16(388)

In [6]:
# PCA Pre-processing
pca = sklearn.decomposition.PCA(n_components=32, whiten=True)
pca.fit(train_features)

# apply the PCA to both train and test features
X_train_pca = pca.transform(train_features)
X_test_pca = pca.transform(test_features)

print(X_train_pca.shape)
print(X_test_pca.shape)

# create new pointers for naming convention's sake
y_train = train_labels
y_test = test_labels

(15707, 32)
(1554, 32)


In [7]:
# Linear classifier using sklearn
linear_model = sklearn.linear_model.LogisticRegression()
linear_model.fit(X_train_pca,y_train)
linear_model_accuracy = linear_model.score(X_test_pca, y_test)
print(f'The linear multi-class classifier has an accuracy of {linear_model_accuracy*100:.2f}%')

The linear multi-class classifier has an accuracy of 83.40%


In [8]:
# Neural Network using sklearn
nn = sklearn.neural_network.MLPClassifier(hidden_layer_sizes=(100,), random_state=1, max_iter=1000, learning_rate_init=0.0025, verbose=True, alpha=0.1)
nn.fit(X_train_pca, y_train)
nn_accuracy = nn.score(X_test_pca, y_test)
print(f'The neural network has a final test accuracy of {nn_accuracy*100:.2f}%')

Iteration 1, loss = 1.10198686
Iteration 2, loss = 0.56255615
Iteration 3, loss = 0.48344804
Iteration 4, loss = 0.44134234
Iteration 5, loss = 0.41237689
Iteration 6, loss = 0.39131708
Iteration 7, loss = 0.37250246
Iteration 8, loss = 0.36021761
Iteration 9, loss = 0.34807679
Iteration 10, loss = 0.33711313
Iteration 11, loss = 0.32878084
Iteration 12, loss = 0.32152963
Iteration 13, loss = 0.31636360
Iteration 14, loss = 0.31053654
Iteration 15, loss = 0.30333158
Iteration 16, loss = 0.29929284
Iteration 17, loss = 0.29425334
Iteration 18, loss = 0.29131487
Iteration 19, loss = 0.28878144
Iteration 20, loss = 0.28505136
Iteration 21, loss = 0.28102722
Iteration 22, loss = 0.27992803
Iteration 23, loss = 0.27592675
Iteration 24, loss = 0.27461242
Iteration 25, loss = 0.27198074
Iteration 26, loss = 0.27002863
Iteration 27, loss = 0.26663247
Iteration 28, loss = 0.26706586
Iteration 29, loss = 0.26430259
Iteration 30, loss = 0.26261709
Iteration 31, loss = 0.25998710
Iteration 32, los

When increasing the max number of iterations during the NN training, this only decreased the NN accuracy on the test set. This suggested that during the training process the NN was overfitting to the training data. Thus, I kept the number of max training iterations high but significantly increased the alpha parameter in order to increase the magnitude of regularization in an attempt to reduce the higher order terms' coefficients and thus reduce overfitting, which seemed to work, boosting the accuracy above the linear model's, being about 85% and 83% respectively.

In [9]:
# Classifier using PyTorch
# turn dataset into tensors
X_train_pca_T = torch.tensor(X_train_pca).float()
X_test_pca_T = torch.tensor(X_test_pca).float()
y_train_T = torch.tensor(y_train)
y_test_T = torch.tensor(y_test)

# load data into torch data handler objects
trainset = TensorDataset(X_train_pca_T, y_train_T)
trainloader = DataLoader(trainset, batch_size=32, shuffle=True)

testset = TensorDataset(X_test_pca_T, y_test_T)
testloader = DataLoader(testset, batch_size=32, shuffle=False)

In [10]:
def calc_torch_accuracy(model: torch.nn.Sequential, dataloader: DataLoader) -> float:
    n_correct: int = 0

    model.eval()
    for X_batch, y_batch in dataloader:
        z_batch = model(X_batch)
        y_predict = torch.argmax(z_batch, dim=1)
        n_correct += torch.sum(y_predict == y_batch)

    n: int = len(dataloader.dataset.tensors[0])
    return float(n_correct / n)

In [11]:
def train_torch_nn(model: torch.nn.Sequential, trainloader: DataLoader, testloader: DataLoader, loss_fn: torch.nn.CrossEntropyLoss, optimizer: torch.optim.SGD, epochs: int=100) -> None:
    model.train()
    for i in range(epochs):
        for X_batch, y_batch in trainloader:
            optimizer.zero_grad()
            z_batch = model(X_batch)
            loss = loss_fn(z_batch, y_batch)
            loss.backward()
            optimizer.step()

        print(f'Epoch {i:4d} - Loss: {loss.item():.5}')

    # calc and display accuracies
    train_acc = calc_torch_accuracy(model, trainloader)
    test_acc = calc_torch_accuracy(model, testloader)
    print(f'Final Training Accuracy: {train_acc:.5f}, Test Accuracy: {test_acc:.5f}')

In [12]:
input_size = X_train_pca.shape[1]
output_size = max(y_train)+1  # number of classes

In [13]:
# PyTorch Linear Classifier Implementation
torch_lin = torch.nn.Sequential(
    torch.nn.Linear(input_size, output_size)
)

lin_loss_fn = torch.nn.CrossEntropyLoss()
lin_optimizer = torch.optim.SGD(torch_lin.parameters(), lr=0.01, weight_decay=0.001)

torch_lin.train()
train_torch_nn(
    model=torch_lin,
    trainloader=trainloader,
    testloader=testloader,
    loss_fn=lin_loss_fn,
    optimizer=lin_optimizer,
    epochs=100
)

Epoch    0 - Loss: 1.0598
Epoch    1 - Loss: 0.66342
Epoch    2 - Loss: 0.92046
Epoch    3 - Loss: 0.92438
Epoch    4 - Loss: 0.38305
Epoch    5 - Loss: 0.57948
Epoch    6 - Loss: 0.76024
Epoch    7 - Loss: 0.7214
Epoch    8 - Loss: 0.50032
Epoch    9 - Loss: 0.54588
Epoch   10 - Loss: 0.64701
Epoch   11 - Loss: 0.41852
Epoch   12 - Loss: 0.83973
Epoch   13 - Loss: 0.49959
Epoch   14 - Loss: 0.85637
Epoch   15 - Loss: 0.51049
Epoch   16 - Loss: 0.63431
Epoch   17 - Loss: 0.59204
Epoch   18 - Loss: 0.589
Epoch   19 - Loss: 0.5439
Epoch   20 - Loss: 0.57088
Epoch   21 - Loss: 0.40623
Epoch   22 - Loss: 0.5671
Epoch   23 - Loss: 0.60278
Epoch   24 - Loss: 0.42306
Epoch   25 - Loss: 1.2848
Epoch   26 - Loss: 0.78871
Epoch   27 - Loss: 0.49422
Epoch   28 - Loss: 0.53169
Epoch   29 - Loss: 0.75213
Epoch   30 - Loss: 0.66736
Epoch   31 - Loss: 0.76914
Epoch   32 - Loss: 0.45047
Epoch   33 - Loss: 0.46213
Epoch   34 - Loss: 0.83145
Epoch   35 - Loss: 0.64185
Epoch   36 - Loss: 0.46142
Epoch   

In [14]:
# PyTorch NN Implementation
hidden_size = 100

torch_nn = torch.nn.Sequential(
                torch.nn.Linear(input_size, hidden_size),
                torch.nn.ReLU(),
                torch.nn.Linear(hidden_size, output_size)
    )
print(torch_nn)

nn_loss_fn = torch.nn.CrossEntropyLoss()
nn_optimizer = torch.optim.SGD(torch_nn.parameters(), lr=0.01, weight_decay=0.001)

torch_nn.train()
train_torch_nn(
    model=torch_nn,
    trainloader=trainloader,
    testloader=testloader,
    loss_fn=nn_loss_fn,
    optimizer=nn_optimizer,
    epochs=100
)

Sequential(
  (0): Linear(in_features=32, out_features=100, bias=True)
  (1): ReLU()
  (2): Linear(in_features=100, out_features=8, bias=True)
)
Epoch    0 - Loss: 1.3316
Epoch    1 - Loss: 0.82039
Epoch    2 - Loss: 0.76706
Epoch    3 - Loss: 0.52838
Epoch    4 - Loss: 0.85729
Epoch    5 - Loss: 0.50733
Epoch    6 - Loss: 0.38612
Epoch    7 - Loss: 0.37146
Epoch    8 - Loss: 0.3726
Epoch    9 - Loss: 0.2551
Epoch   10 - Loss: 0.46443
Epoch   11 - Loss: 0.61359
Epoch   12 - Loss: 0.41107
Epoch   13 - Loss: 0.35155
Epoch   14 - Loss: 0.35183
Epoch   15 - Loss: 0.11199
Epoch   16 - Loss: 0.5458
Epoch   17 - Loss: 0.182
Epoch   18 - Loss: 0.52823
Epoch   19 - Loss: 0.47726
Epoch   20 - Loss: 0.41967
Epoch   21 - Loss: 0.4453
Epoch   22 - Loss: 0.25493
Epoch   23 - Loss: 0.43893
Epoch   24 - Loss: 0.65242
Epoch   25 - Loss: 0.3039
Epoch   26 - Loss: 0.52203
Epoch   27 - Loss: 0.31335
Epoch   28 - Loss: 0.33027
Epoch   29 - Loss: 0.17789
Epoch   30 - Loss: 0.36497
Epoch   31 - Loss: 0.58035

In [15]:
# Final model comparison
print('\nSklearn Models')
print(f'Linear Final Training Accuracy: {linear_model.score(X_train_pca, y_train):.5f}, Test Accuracy: {linear_model.score(X_test_pca, y_test):.5f}')
print(f'NN Final Training Accuracy: {nn.score(X_train_pca, y_train):.5f}, Test Accuracy: {nn.score(X_test_pca, y_test):.5f}')

print('\nPyTorch Models')
print(f'Linear Final Training Accuracy: {calc_torch_accuracy(torch_lin, trainloader):.5f}, Test Accuracy: {calc_torch_accuracy(torch_lin, testloader):.5f}')
print(f'NN Final Training Accuracy: {calc_torch_accuracy(torch_nn, trainloader):.5f}, Test Accuracy: {calc_torch_accuracy(torch_nn, testloader):.5f}')


Sklearn Models
Linear Final Training Accuracy: 0.85535, Test Accuracy: 0.83398
NN Final Training Accuracy: 0.96931, Test Accuracy: 0.85135

PyTorch Models
Linear Final Training Accuracy: 0.84822, Test Accuracy: 0.82754
NN Final Training Accuracy: 0.92118, Test Accuracy: 0.86229


### Code Explanation
Sources: I mostly used the sklearn and pytorch documentation/APIs for guidance when completing this assignment, especially the examples for how the different classes and functions I needed were used, as well as which classes and functions were even available. For example, using the sklearn docs to understand the difference between PCA.fit() and PCA.transform, or looking at the examples in the torch.nn.Sequential() docs when implementing my models with PyTorch. That being said, I took a glance at Professor Ventura's paper on this topic and the linked Github with the code used in order to double check that I was applying my PCA correctly, as the dimensions of the matrix returned by the PCA.fit() function initially caused me some confusion. I also used Perplexity (AI) for double checking how to use torch.nn.Sequential() for just a linear multiclass classifier, and further explanation of several things like what loss_fn.backward() is doing under the hood, why the PyTorch docs uses iter() in its DataLoader usage examples, among other things.

### Discussion
Each row correponds to a data point, while the columns correspond to the data points' features. The ranges of the labels are [0, 7], corresponding the the 8 classes in this classification problem, each of which correspond to a possible tree species.